In [ ]:


import os, math
import numpy as np
from PIL import Image
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from fastai.vision.all import *

Notebook for Inference: Testing Individual Images with Visualisation

In [ ]:
# ========== Configuration ==========

BMP_PATH        = "../data/test/images/pcb_0006_jpg.rf.3643e486b19e08c1dd189e7ac8bd3bbb.jpg"  # insert image path
PNG_OUT_PATH    = "./tested_image.png"  # insert output path for PNG-output
MODEL_NAME      = "unet_trained"                     # insert saved model without .pth ending
MODEL_DIR       = 'models'                    # insert folder, where your model is saved
PATCH_SIZE      = (512, 512)                  # Patch size (H, W)
NUM_CLASSES     = 2                           # as in your Training (n_out=2)
BATCH_SIZE      = 8                           # Batch size
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
GT_MASK_PATH    = None                        # Optional: Ground-Truth-Mask 
SAVE_PRED_MASK  = PNG_OUT_PATH.replace(".png", "_mask_prediction.png")       # output path for prediction
OVERLAY_ALPHA   = 0.35                        # Transparency in visualisation overlay
# ===================================


In [ ]:
def load_bmp_as_gray_and_save_png(bmp_path, png_path):
    img = Image.open(bmp_path).convert("L")  
    img.save(png_path)
    return np.array(img)  

In [ ]:
def pad_to_multiple(img_np, patch_size):
    H, W = img_np.shape
    ph, pw = patch_size
    H_pad = math.ceil(H / ph) * ph
    W_pad = math.ceil(W / pw) * pw
    pad_h = H_pad - H
    pad_w = W_pad - W
    img_pad = np.pad(img_np, ((0, pad_h), (0, pad_w)), mode="reflect")
    return img_pad, (H, W)

In [ ]:
def extract_patches(img_np, patch_size):
    ph, pw = patch_size
    H, W = img_np.shape
    patches, coords = [], []
    for y in range(0, H, ph):
        for x in range(0, W, pw):
            patch = img_np[y:y+ph, x:x+pw]
            patches.append(patch)
            coords.append((y, x))
    return patches, coords, (H, W)

In [ ]:
def reconstruct_from_patches(patches, coords, full_shape):
    H, W = full_shape
    canvas = np.zeros((H, W), dtype=patches[0].dtype)
    ph, pw = patches[0].shape
    for patch, (y, x) in zip(patches, coords):
        canvas[y:y+ph, x:x+pw] = patch
    return canvas

In [ ]:
def colorize_multiclass(mask_np, num_classes):
    cmap = plt.get_cmap('tab20')
    colored = cmap((mask_np % num_classes) / max(num_classes-1, 1))
    return (colored[..., :3] * 255).astype(np.uint8)

In [ ]:
def compute_metrics(pred_mask_np, gt_mask_np, num_classes=2):
    assert pred_mask_np.shape == gt_mask_np.shape
    dices, ious = [], []
    for c in range(num_classes):
        pred_c = (pred_mask_np == c)
        gt_c   = (gt_mask_np == c)
        inter  = np.logical_and(pred_c, gt_c).sum()
        union  = np.logical_or(pred_c, gt_c).sum()
        iou_c  = inter / (union + 1e-8)
        dice_c = (2 * inter) / (pred_c.sum() + gt_c.sum() + 1e-8)
        ious.append(iou_c)
        dices.append(dice_c)
    acc = (pred_mask_np == gt_mask_np).mean()
    return {"accuracy": acc, "mIoU": float(np.mean(ious)), "mDice": float(np.mean(dices))}


In [ ]:
print(f"Device: {DEVICE}")
# load image and convert to grayscale
img_np = load_bmp_as_gray_and_save_png(BMP_PATH, PNG_OUT_PATH)
H_orig, W_orig = img_np.shape
print(f"Originalgröße: {img_np.shape}")

In [ ]:
# Padding + Patches
img_pad, orig_shape = pad_to_multiple(img_np, PATCH_SIZE)
patches, coords, full_shape = extract_patches(img_pad, PATCH_SIZE)
print(f"Anzahl Patches: {len(patches)} | gepaddete Größe: {full_shape}")


In [ ]:
# Dummy-Batch for fastai
dummy_x = torch.zeros(1, 3, PATCH_SIZE[0], PATCH_SIZE[1])
dummy_y = torch.zeros(1, PATCH_SIZE[0], PATCH_SIZE[1]).long()

dls = DataLoaders.from_dsets([(dummy_x[0], dummy_y[0])], [(dummy_x[0], dummy_y[0])], bs=1, device=DEVICE)

In [ ]:
# DataLoaders
learn = unet_learner(dls, resnet34, n_out=NUM_CLASSES, pretrained=False, loss_func=CrossEntropyLossFlat(axis=1))
learn.model_dir = 'models' 
learn.load(MODEL_NAME)                    
model = learn.model.eval().to(DEVICE)

# Estimated Time: 2m 22.9s

In [ ]:
# Normalisation
mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1).to(DEVICE)
std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1).to(DEVICE)


In [ ]:
# Inference for Patches (Batch-compatible)
preds_list = []
with torch.no_grad():
    for i in range(0, len(patches), BATCH_SIZE):
        batch_np = patches[i:i+BATCH_SIZE]
        
        batch_3ch = [np.stack([p]*3, axis=-1) for p in batch_np]                 
        batch_t   = torch.stack([torch.from_numpy(p).permute(2,0,1)              
                                     for p in batch_3ch]).float() / 255.0            
        batch_t   = ((batch_t.to(DEVICE) - mean) / std)
        logits    = model(batch_t)                                               
        pred      = torch.argmax(logits, dim=1).cpu().numpy()                    
        preds_list.extend(pred)

In [ ]:
# Assemble the prediction & crop it back to its original size
pred_canvas = reconstruct_from_patches(preds_list, coords, full_shape)
pred_final  = pred_canvas[:H_orig, :W_orig]

In [ ]:

def ensure_dir(path):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)


In [ ]:
def colorize_multiclass_custom(mask_np, colors):
    
    h, w = mask_np.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in enumerate(colors):
        out[mask_np == cls] = color
    return out

# set colours
custom_colors = [
    (0, 0, 0),       # class 0 = black (background)
    (255, 0, 0),     # class 1 = red
    (0, 255, 0),     # class 2 = green (if needed)
    (0, 0, 255),     # class 3 = blue (if needed)
]

colored = colorize_multiclass_custom(pred_final, custom_colors)


In [ ]:
SHOW_PLOT = True
DOWNSAMPLE_FACTOR = 4  # Downsampling factor for display
OVERLAY_ALPHA = 0.5    # transparency for overlay

colored = colorize_multiclass_custom(pred_final, custom_colors)

# Downsampling
display_img = img_np[::DOWNSAMPLE_FACTOR, ::DOWNSAMPLE_FACTOR]
display_pred = colored[::DOWNSAMPLE_FACTOR, ::DOWNSAMPLE_FACTOR]

if SHOW_PLOT:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Subplot 1: Original
    axes[0].imshow(display_img, cmap='gray')
    axes[0].set_title("Original (Downsampled)")
    axes[0].axis('off')

    # Subplot 2: Prediction
    axes[1].imshow(display_pred)
    axes[1].set_title(f"Prediction (Downsampled, Classes={NUM_CLASSES})")
    axes[1].axis('off')

    # Subplot 3: Overlay (Original + Prediction)
    axes[2].imshow(display_img, cmap='gray')
    axes[2].imshow(display_pred, alpha=OVERLAY_ALPHA)
    axes[2].set_title("Overlay (Downsampled)")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
    plt.close('all')


In [ ]:
# Crop-Image showing a section in full resolution
CROP_X = 2600      # left corner in pixels
CROP_Y = 1380       # top corner in pixels
CROP_W = 1200    # width of the crop region
CROP_H = 1200    # height of the crop region
MAX_PLOT_PIXELS = 1_500_000  # if too large, don't display but save
SAVE_CROPPED_FULLRES = True

h, w = img_np.shape
x0 = min(max(0, CROP_X), w - 1)
y0 = min(max(0, CROP_Y), h - 1)
x1 = min(x0 + CROP_W, w)
y1 = min(y0 + CROP_H, h)

img_crop = img_np[y0:y1, x0:x1]
pred_crop = colored[y0:y1, x0:x1]
area = img_crop.shape[0] * img_crop.shape[1]
print(f"Crop region: x={x0}:{x1}, y={y0}:{y1}, size={img_crop.shape}")

if area == 0:
    raise ValueError("Crop region is empty. Please adjust CROP_X, CROP_Y, CROP_W, CROP_H.")

can_show = SHOW_PLOT and area <= MAX_PLOT_PIXELS
if can_show:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img_crop, cmap='gray')
    axes[0].set_title("Original Image")
    axes[0].axis('off')

    axes[1].imshow(pred_crop)
    axes[1].set_title(f"Prediction, Classes={NUM_CLASSES}")
    axes[1].axis('off')

    axes[2].imshow(img_crop, cmap='gray')
    axes[2].imshow(pred_crop, alpha=OVERLAY_ALPHA)
    axes[2].set_title("Overlay")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
    plt.close('all')
else:
    print(f"Crop too large to display ({area} pixels). Saving crop images instead.")

if SAVE_CROPPED_FULLRES:
    crop_base = SAVE_PRED_MASK.replace(".png", f"_crop_{x0}_{y0}_{x1}_{y1}")
    ensure_dir(crop_base)

    Image.fromarray(img_crop).save(crop_base + "_image.png")
    Image.fromarray(pred_crop).save(crop_base + "_prediction.png")

    overlay_crop = np.stack([img_crop] * 3, axis=-1).astype(np.float32)
    overlay_crop = ((overlay_crop * (1.0 - OVERLAY_ALPHA)) + (pred_crop.astype(np.float32) * OVERLAY_ALPHA)).clip(0, 255).astype(np.uint8)
    Image.fromarray(overlay_crop).save(crop_base + "_overlay.png")

    print(f"Crop saved: {crop_base}_image.png")
    print(f"Prediction saved: {crop_base}_prediction.png")
    print(f"Overlay saved: {crop_base}_overlay.png")

In [ ]:
# save predicition
# save overlay as separate file
Image.fromarray(colored).save(SAVE_PRED_MASK)
print(f"Prediction saved: {SAVE_PRED_MASK}")

In [ ]:
# optionally compute metrics if GT is provided
if GT_MASK_PATH and os.path.exists(GT_MASK_PATH):
    gt = np.array(Image.open(GT_MASK_PATH).convert("L"))
    gt = gt[:H_orig, :W_orig]  # falls GT größer, croppen
    metrics = compute_metrics(pred_final.astype(np.int32), gt.astype(np.int32), num_classes=NUM_CLASSES)
    print(f"Pixel-Acc: {metrics['accuracy']:.4f} | mIoU: {metrics['mIoU']:.4f} | mDice: {metrics['mDice']:.4f}")
else:
    print("No Ground-Truth mask provided or file does not exist. Metrics not computed.")
